In [1]:
# read data
import json
import pandas as pd

articles = pd.read_csv('Flatfeet_clean.csv')
columns = ['Title', 'Author', 'Publication Year', 'Abstract Note', 'Journal Abbreviation']
articles = articles[columns]
articles['id'] = list(range(1, len(articles) + 1))

abstracts = articles['Abstract Note'].tolist()
abstract_dict = dict(zip(articles['id'], abstracts))
article_ids = articles['id'].tolist()


In [2]:
from gliner2 import GLiNER2

# Load the model
extractor = GLiNER2.from_pretrained('fastino/gliner2-large-v1')


2026-03-18 23:41:50.179618406 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
/home/homeless/.virtualenvs/ts_lab/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-large
Counting layer     : count_lstm
Token pooling      : first


In [3]:
# round 1 entity extraction
all_results = {}

fields = [
    'Anatomical Entity',
    'Symptom',
    'Terms of Body Movements',
    'Population',
]

round_one_schema = {
    'Anatomical Entity': 'Any structure within a biological organism.',
    'Symptom': 'Medical symptoms, conditions, and clinically relevant complaints.',
    'Terms of Body Movements': "Movements, biomechanical measures, postures, alignments, and motion phrases like 'ankle dorsiflexion' or 'hip internal rotation'.",
    'Population': 'The population of interest, including age, sex, sample size, clinical status, and study groups.',
}

round_two_schemas = {
    'Symptom': {
        'anatomical structure': 'Where is the symptom located?',
        'symptom': 'What is the symptom, diagnosis, or condition?',
    },
    'Terms of Body Movements': {
        'anatomical structure': 'Which anatomical structure is involved in the movement?',
        'movement': 'What is the movement, posture, alignment, or biomechanical measure?',
    },
    'Population': {
        'age': 'Age, age range, or developmental stage.',
        'level': 'Athelete level, activity level, fitness level, or non-athlete.',
        'clinical status': 'Clinical condition, case/control status, or disease status.',
        'sex': 'Sex or gender.',
        'study group': 'Named cohort, subgroup, or study arm.',
    },
}


def extract_round_one_entities(text):
    return extractor.extract_entities(text, round_one_schema).get('entities', {})


for i, text in enumerate(abstracts, start=1):
    if pd.isna(text) or str(text).strip() == '':
        print(f'[{i}] skipped: empty abstract')
        all_results[str(i)] = {}
        continue

    all_results[str(i)] = extract_round_one_entities(text)

with open('entities_pretrainedmodel.json', 'w') as f:
    json.dump(all_results, f, indent=4)

print(f"Saved {len(all_results)} abstracts to entities_pretrainedmodel.json")


Saved 19 abstracts to entities_pretrainedmodel.json


In [5]:
# entity normalization helpers
import re
from collections import defaultdict

ABBREVIATION_MAP = {
    'AP': 'anterior-posterior',
    'BF': 'biceps femoris',
    'BFFG': 'bilateral flexible flat feet group',
    'KF': 'knee flexion',
    'LBP': 'low back pain',
    'MG': 'medial gastrocnemius',
    'MLBP': 'mechanical low back pain',
    'MTSS': 'medial tibial stress syndrome',
    'NFG': 'normal foot group',
    'OA': 'osteoarthritis',
    'OFFG': 'one-sided flexible flat feet group',
    'PF': 'plantar fasciitis',
    'PFP': 'patellofemoral pain',
    'PFPS': 'patellofemoral pain syndrome',
    'RF': 'rectus femoris',
    'TA': 'tibialis anterior',
}

IRREGULAR_SINGULARS = {
    'feet': 'foot',
    'teeth': 'tooth',
    'men': 'man',
    'women': 'woman',
    'children': 'child',
    'people': 'person',
    'indices': 'index',
}

SINGULAR_OVERRIDES = {
    'alignments': 'alignment',
    'analyses': 'analysis',
    'angles': 'angle',
    'areas': 'area',
    'changes': 'change',
    'controls': 'control',
    'deformations': 'deformation',
    'disorders': 'disorder',
    'extremities': 'extremity',
    'factors': 'factor',
    'flexions': 'flexion',
    'grades': 'grade',
    'imbalances': 'imbalance',
    'individuals': 'individual',
    'joints': 'joint',
    'knees': 'knee',
    'limbs': 'limb',
    'measures': 'measure',
    'motions': 'motion',
    'muscles': 'muscle',
    'parameters': 'parameter',
    'patients': 'patient',
    'pathologies': 'pathology',
    'planes': 'plane',
    'postures': 'posture',
    'pressures': 'pressure',
    'referees': 'referee',
    'runners': 'runner',
    'scores': 'score',
    'subjects': 'subject',
    'symptoms': 'symptom',
    'torsions': 'torsion',
    'volunteers': 'volunteer',
}

TOKEN_SPLIT_PATTERN = re.compile(r'([\s/(),]+)')


def _tokenize_with_spans(text):
    return list(re.finditer(r'\S+', text))


def _char_to_start_token(token_matches, start_char):
    start_token = None
    for token_idx, token in enumerate(token_matches):
        if start_token is None and token.start() <= start_char < token.end():
            start_token = token_idx
            break
    return start_token


def _collect_match_positions(text, token_matches, entity_text, ignore_case=False):
    flags = re.IGNORECASE if ignore_case else 0
    positions = []
    for match in re.finditer(re.escape(entity_text), text, flags=flags):
        start_token = _char_to_start_token(token_matches, match.start())
        if start_token is not None:
            positions.append(start_token)
    return positions


def _is_abbreviation(text):
    letters = [char for char in text if char.isalpha()]
    return bool(letters) and all(char.isupper() for char in letters)


def _singularize_word(word):
    if not word or _is_abbreviation(word):
        return word
    lower_word = word.lower()
    if lower_word in IRREGULAR_SINGULARS:
        return IRREGULAR_SINGULARS[lower_word]
    if lower_word in SINGULAR_OVERRIDES:
        return SINGULAR_OVERRIDES[lower_word]
    if len(lower_word) <= 3:
        return lower_word
    if lower_word.endswith('ies') and len(lower_word) > 4:
        return lower_word[:-3] + 'y'
    if lower_word.endswith('sses') or lower_word.endswith('xes') or lower_word.endswith('zes'):
        return lower_word[:-2]
    if lower_word.endswith('s') and not lower_word.endswith('ss'):
        return lower_word[:-1]
    return lower_word


def _expand_abbreviation_token(token):
    if not token or TOKEN_SPLIT_PATTERN.fullmatch(token):
        return token
    if _is_abbreviation(token):
        return ABBREVIATION_MAP.get(token, token.lower())
    return token.lower()


def _normalize_token(token):
    if not token or TOKEN_SPLIT_PATTERN.fullmatch(token):
        return token
    return _singularize_word(token)


def _normalize_hyphen_spacing(text):
    return re.sub(r'(?<!\d)-(?!\d)|(?<!\d)-(?=\d)|(?<=\d)-(?!\d)', ' ', text)


def _normalize_canonical_form(text, field):
    if not text or not text.strip():
        return None

    normalized = _normalize_hyphen_spacing(text.strip())
    normalized = ''.join(_expand_abbreviation_token(part) for part in TOKEN_SPLIT_PATTERN.split(normalized))
    normalized = ''.join(_normalize_token(part) for part in TOKEN_SPLIT_PATTERN.split(normalized))
    normalized = re.sub(r'\s+', ' ', normalized).strip(' -/(),')

    if field == 'Anatomical Entity':
        if normalized == 'joint':
            return None
        if normalized.endswith(' joint'):
            normalized = normalized[:-6].strip()
            if not normalized:
                return None
        if normalized.startswith('left '):
            normalized = normalized[5:].strip()
        elif normalized.startswith('right '):
            normalized = normalized[6:].strip()
        if not normalized:
            return None

    return normalized or None


In [3]:
# normalize entities

def build_canonical_entities(abstracts_dict, entity_dict):
    canonical_records = []
    canonical_index = {}

    for abstract_id in sorted(abstracts_dict):
        text = '' if abstracts_dict[abstract_id] is None else str(abstracts_dict[abstract_id])
        abstract_entities = entity_dict.get(str(abstract_id), entity_dict.get(abstract_id, {}))
        token_matches = _tokenize_with_spans(text)

        exact_match_cache = {}
        ignorecase_match_cache = {}

        for field, entities in abstract_entities.items():
            seen_entity_texts = set()
            for entity_text in entities:
                if not isinstance(entity_text, str) or not entity_text.strip():
                    continue

                entity_text = entity_text.strip()
                if entity_text in seen_entity_texts:
                    continue
                seen_entity_texts.add(entity_text)

                if entity_text not in exact_match_cache:
                    exact_match_cache[entity_text] = _collect_match_positions(text, token_matches, entity_text)
                match_positions = exact_match_cache[entity_text]

                if match_positions:
                    positions = match_positions
                else:
                    fallback_key = entity_text.casefold()
                    if fallback_key not in ignorecase_match_cache:
                        ignorecase_match_cache[fallback_key] = _collect_match_positions(
                            text,
                            token_matches,
                            entity_text,
                            ignore_case=True,
                        )
                    positions = ignorecase_match_cache[fallback_key]

                if not positions:
                    positions = [None]

                canonical_key = (field, entity_text)
                if canonical_key not in canonical_index:
                    canonical_index[canonical_key] = len(canonical_records)
                    canonical_records.append({
                        'canonical_id': f'ent_{len(canonical_records) + 1:03d}',
                        'canonical_form': entity_text,
                        'field': field,
                        'variants': [entity_text],
                        'occurrences': [],
                    })

                for position in positions:
                    canonical_records[canonical_index[canonical_key]]['occurrences'].append({
                        'abstract_id': f'abs_{int(abstract_id):03d}',
                        'original_text': entity_text,
                        'position': position,
                    })

    return canonical_records


def normalize_canonical_entities(canonical_entities):
    merged_records = {}

    for record in canonical_entities:
        normalized_form = _normalize_canonical_form(record['canonical_form'], record['field'])
        if normalized_form is None:
            continue

        merge_key = (record['field'], normalized_form)
        if merge_key not in merged_records:
            merged_records[merge_key] = {
                'canonical_form': normalized_form,
                'field': record['field'],
                'variants': [],
                'occurrences': [],
            }

        merged_record = merged_records[merge_key]
        for variant in record.get('variants', []):
            if variant not in merged_record['variants']:
                merged_record['variants'].append(variant)

        merged_record['occurrences'].extend(record.get('occurrences', []))

    normalized_records = []
    for idx, merged_record in enumerate(sorted(merged_records.values(), key=lambda item: (item['field'], item['canonical_form'])), start=1):
        normalized_records.append({
            'canonical_id': f'ent_{idx:03d}',
            'canonical_form': merged_record['canonical_form'],
            'field': merged_record['field'],
            'variants': merged_record['variants'],
            'occurrences': merged_record['occurrences'],
        })

    return normalized_records


def build_abstract_aggregated_entities(canonical_entities, abstracts_dict):
    abstract_records = {
        f'abs_{int(abstract_id):03d}': {
            'abstract': {
                'abstract_id': f'abs_{int(abstract_id):03d}',
                'text': '' if abstract_text is None else str(abstract_text),
            },
            'entities': [],
            'output_format': {
                'triples': [],
            },
        }
        for abstract_id, abstract_text in abstracts_dict.items()
    }

    for entity in canonical_entities:
        for occurrence in entity.get('occurrences', []):
            abstract_id = occurrence['abstract_id']
            abstract_records[abstract_id]['entities'].append({
                'entity_id': entity['canonical_id'],
                'entity_form': entity['canonical_form'],
                'field': entity['field'],
                'mention': {
                    'original_text': occurrence['original_text'],
                    'position': occurrence['position'],
                },
            })

    for record in abstract_records.values():
        record['entities'].sort(
            key=lambda item: (
                float('inf') if item['mention']['position'] is None else item['mention']['position'],
                item['entity_id'],
            )
        )

    return list(abstract_records.values())


def build_normalized_abstract_entities(canonical_entities_raw, abstracts_dict):
    normalized_entities = normalize_canonical_entities(canonical_entities_raw)
    return build_abstract_aggregated_entities(normalized_entities, abstracts_dict)



In [ ]:

canonical_entities_raw = build_canonical_entities(abstract_dict, all_results)
abstract_aggregated_entities = build_normalized_abstract_entities(canonical_entities_raw, abstract_dict)

with open('normalized_entities.json', 'w') as f:
    json.dump({'abstracts': abstract_aggregated_entities}, f, indent=4)

print(f"Saved {len(abstract_aggregated_entities)} abstracts to normalized_entities.json")



In [6]:
# method 1
# second round extraction on normalized entity forms

def build_round_two_lookup(abstract_aggregated_entities):
    round_two_lookup = {field: {} for field in round_two_schemas}

    for record in abstract_aggregated_entities:
        for entity in record['entities']:
            field = entity['field']
            entity_form = entity['entity_form']
            if field not in round_two_schemas:
                continue
            if entity_form not in round_two_lookup[field]:
                round_two_lookup[field][entity_form] = extractor.extract_entities(
                    entity_form,
                    round_two_schemas[field],
                ).get('entities', {})

    return {field: lookup for field, lookup in round_two_lookup.items() if lookup}


round_two_lookup = build_round_two_lookup(abstract_aggregated_entities)
normalized_entity_data = {
    'round2_lookup': round_two_lookup,
    'abstracts': abstract_aggregated_entities,
}

with open('entities_2nd_pretrainedmodel.json', 'w') as f:
    json.dump(normalized_entity_data, f, indent=4)

print(f"Saved round 2 lookup for {sum(len(v) for v in round_two_lookup.values())} normalized entity forms")



Saved round 2 lookup for 148 normalized entity forms


In [6]:
# method 2

import pandas as pd
import hashlib
import numpy as np
import json

top_k = 1
embedding_dim = 512


with open('entities_2nd_pretrainedmodel.json', 'r') as f:
    method1_data = json.load(f)


def normalize_eval_phrase(text, field):
    if pd.isna(text) or not str(text).strip():
        return None
    return _normalize_canonical_form(str(text), field)


def normalize_anatomical_location(text):
    if pd.isna(text) or not str(text).strip():
        return None
    return _normalize_canonical_form(str(text), 'Anatomical Entity')


def normalize_method1_predictions(attributes):
    predictions = []
    for item in attributes.get('anatomical structure', []):
        normalized_item = normalize_anatomical_location(item)
        if normalized_item and normalized_item not in predictions:
            predictions.append(normalized_item)
    return predictions


def hash_feature(feature, dim):
    digest = hashlib.md5(feature.encode('utf-8')).hexdigest()
    return int(digest, 16) % dim


def build_text_embedding(text, dim=512):
    text = re.sub(r'\s+', ' ', text.lower()).strip()
    vector = np.zeros(dim, dtype=float)
    if not text:
        return vector

    for token in text.split():
        vector[hash_feature(f'tok:{token}', dim)] += 1.0

    padded = f'  {text} '
    for i in range(len(padded) - 2):
        trigram = padded[i:i + 3]
        vector[hash_feature(f'tri:{trigram}', dim)] += 1.0

    norm = np.linalg.norm(vector)
    return vector / norm if norm else vector


def rank_anatomical_candidates(query_text, candidate_forms, top_k=1, dim=512):
    if not query_text or not candidate_forms:
        return []

    query_embedding = build_text_embedding(query_text, dim)
    scored_candidates = []
    for candidate in candidate_forms:
        candidate_embedding = build_text_embedding(candidate, dim)
        score = float(np.dot(query_embedding, candidate_embedding))
        scored_candidates.append((score, candidate))

    scored_candidates.sort(key=lambda item: (-item[0], item[1]))
    return [candidate for _, candidate in scored_candidates[:top_k]]


abstract_entity_records = method1_data['abstracts']
abstract_anatomy_lookup = {}
for record in abstract_entity_records:
    abstract_id = record['abstract']['abstract_id']
    anatomy_forms = []
    for entity in record['entities']:
        if entity['field'] != 'Anatomical Entity':
            continue
        anatomy_form = normalize_anatomical_location(entity['entity_form'])
        if anatomy_form and anatomy_form not in anatomy_forms:
            anatomy_forms.append(anatomy_form)
    abstract_anatomy_lookup[abstract_id] = anatomy_forms

method2_cases = []
for record in abstract_entity_records:
    abstract_id = record['abstract']['abstract_id']
    candidate_forms = abstract_anatomy_lookup.get(abstract_id, [])
    for entity in record['entities']:
        if entity['field'] not in {'Symptom', 'Terms of Body Movements'}:
            continue
        predictions = rank_anatomical_candidates(
            entity['entity_form'],
            candidate_forms,
            top_k=top_k,
            dim=embedding_dim,
        )
        method2_cases.append({
            'abstract_id': abstract_id,
            'entity_form': entity['entity_form'],
            'field': entity['field'],
            'mention': entity['mention'],
            'candidate_anatomies': candidate_forms,
            'predicted_locations': predictions,
        })

print(f'Method 2 generated {len(method2_cases)} cases.')

with open('entity_specification.json', 'w') as f:
    json.dump(method2_cases, f, indent=4)




Method 2 generated 307 cases.


In [8]:
# BERT-based method 2 variants

import torch
from transformers import AutoModel, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
bert_batch_size = 16

bert_model_specs = {
    'biobert': 'dmis-lab/biobert-base-cased-v1.1',
    'pubmedbert': 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext',
    'scibert': 'allenai/scibert_scivocab_uncased',
}

bert_method2_results = {}
bert_method2_cases = {}


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


def encode_texts_with_transformer(texts, tokenizer, model, batch_size=16):
    vectors = {}
    ordered_texts = list(dict.fromkeys(texts))
    if not ordered_texts:
        return vectors

    model.eval()
    with torch.no_grad():
        for start in range(0, len(ordered_texts), batch_size):
            batch = ordered_texts[start:start + batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, return_tensors='pt')
            encoded = {key: value.to(device) for key, value in encoded.items()}
            outputs = model(**encoded)
            pooled = mean_pool(outputs.last_hidden_state, encoded['attention_mask'])
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
            for text, vector in zip(batch, pooled.cpu().numpy()):
                vectors[text] = vector

    return vectors


def rank_anatomical_candidates_with_vectors(query_text, candidate_forms, vector_lookup, top_k=1):
    if not query_text or not candidate_forms or query_text not in vector_lookup:
        return []

    query_vector = vector_lookup[query_text]
    scored = []
    for candidate in candidate_forms:
        if candidate not in vector_lookup:
            continue
        score = float(np.dot(query_vector, vector_lookup[candidate]))
        scored.append((score, candidate))

    scored.sort(key=lambda item: (-item[0], item[1]))
    return [candidate for _, candidate in scored[:top_k]]


all_anatomy_forms = sorted({candidate for candidates in abstract_anatomy_lookup.values() for candidate in candidates})
query_forms = sorted({case['entity_form'] for case in method2_cases})
texts_to_encode = all_anatomy_forms + query_forms

for model_name, model_id in bert_model_specs.items():
    print(f'Loading {model_name}: {model_id} on {device}')
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id, use_safetensors=True).to(device)

    vector_lookup = encode_texts_with_transformer(
        texts_to_encode,
        tokenizer,
        model,
        batch_size=bert_batch_size,
    )

    cases = []
    for case in method2_cases:
        predictions = rank_anatomical_candidates_with_vectors(
            case['entity_form'],
            case['candidate_anatomies'],
            vector_lookup,
            top_k=top_k,
        )
        cases.append({
            'abstract_id': case['abstract_id'],
            'entity_form': case['entity_form'],
            'field': case['field'],
            'mention': case['mention'],
            'candidate_anatomies': case['candidate_anatomies'],
            'predicted_locations': predictions,
        })

    bert_method2_cases[model_name] = cases
    bert_method2_results[model_name] = {
        'model_id': model_id,
        'device': device,
        'cases': cases,
    }

print({model_name: len(result['cases']) for model_name, result in bert_method2_results.items()})
bert_method2_cases



Loading biobert: dmis-lab/biobert-base-cased-v1.1 on cuda


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Loading pubmedbert: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext on cuda


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Loading scibert: allenai/scibert_scivocab_uncased on cuda


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'biobert': 307, 'pubmedbert': 307, 'scibert': 307}


{'biobert': [{'abstract_id': 'abs_001',
   'entity_form': 'flat foot',
   'field': 'Symptom',
   'mention': {'original_text': 'flat foot', 'position': 14},
   'candidate_anatomies': ['medial longitudinal arch',
    'medical longitudinal arch',
    'knee'],
   'predicted_locations': ['medical longitudinal arch']},
  {'abstract_id': 'abs_001',
   'entity_form': 'pronated foot',
   'field': 'Symptom',
   'mention': {'original_text': 'pronated foot', 'position': 19},
   'candidate_anatomies': ['medial longitudinal arch',
    'medical longitudinal arch',
    'knee'],
   'predicted_locations': ['medial longitudinal arch']},
  {'abstract_id': 'abs_001',
   'entity_form': 'anterior knee pain',
   'field': 'Symptom',
   'mention': {'original_text': 'anterior knee pain', 'position': 44},
   'candidate_anatomies': ['medial longitudinal arch',
    'medical longitudinal arch',
    'knee'],
   'predicted_locations': ['medial longitudinal arch']},
  {'abstract_id': 'abs_001',
   'entity_form': 'knee 

In [9]:
# evaluation against golden standard

gold_rows = pd.read_csv('golden_standard.csv')


def location_match(predicted_location, gold_location):
    if not predicted_location or not gold_location:
        return False
    if predicted_location == gold_location:
        return True

    predicted_tokens = set(predicted_location.split())
    gold_tokens = set(gold_location.split())
    return gold_tokens.issubset(predicted_tokens) or predicted_tokens.issubset(gold_tokens)


field_by_type = {
    'symptom': 'Symptom',
    'movement': 'Terms of Body Movements',
}

round_two_lookup = method1_data['round2_lookup']
bert_case_lookup = {}
for model_name, cases in bert_method2_cases.items():
    model_lookup = {}
    for case in cases:
        key = (case['abstract_id'], case['field'], case['entity_form'], case['mention']['position'])
        model_lookup[key] = case['predicted_locations']
    bert_case_lookup[model_name] = model_lookup

evaluation_rows = []

for _, row in gold_rows.iterrows():
    gold_type = str(row['Type']).strip().lower()
    if gold_type not in field_by_type:
        continue

    abstract_id = f"abs_{int(row['Abstract']):03d}"
    field = field_by_type[gold_type]
    query_form = normalize_eval_phrase(row['Full Phrase'], field)
    gold_location = normalize_anatomical_location(row['Anatomical Location'])

    method1_attributes = round_two_lookup.get(field, {}).get(query_form, {})
    method1_predictions = normalize_method1_predictions(method1_attributes)
    method1_correct = any(location_match(prediction, gold_location) for prediction in method1_predictions)

    traditional_predictions = rank_anatomical_candidates(
        query_form,
        abstract_anatomy_lookup.get(abstract_id, []),
        top_k=top_k,
        dim=embedding_dim,
    )
    traditional_correct = any(location_match(prediction, gold_location) for prediction in traditional_predictions)

    bert_predictions = {}
    bert_correct = {}
    for model_name, model_lookup in bert_case_lookup.items():
        matching_predictions = []
        for key, predictions in model_lookup.items():
            case_abstract_id, case_field, case_query_form, _ = key
            if case_abstract_id == abstract_id and case_field == field and case_query_form == query_form:
                matching_predictions = predictions
                break
        bert_predictions[model_name] = matching_predictions
        bert_correct[model_name] = any(location_match(prediction, gold_location) for prediction in matching_predictions)

    evaluation_rows.append({
        'abstract_id': abstract_id,
        'type': gold_type,
        'full_phrase': row['Full Phrase'],
        'gold_location': gold_location,
        'method1_predictions': method1_predictions,
        'method1_correct': method1_correct,
        'traditional_method2_predictions': traditional_predictions,
        'traditional_method2_correct': traditional_correct,
        'bert_predictions': bert_predictions,
        'bert_correct': bert_correct,
    })


def summarize_method(evaluation_rows, prediction_key, correct_key):
    total = len(evaluation_rows)
    predicted = sum(1 for row in evaluation_rows if row[prediction_key])
    correct = sum(1 for row in evaluation_rows if row[correct_key])
    return {
        'total': total,
        'predicted': predicted,
        'coverage': (predicted / total * 100) if total else 0,
        'correct': correct,
        'accuracy': (correct / predicted * 100) if predicted else 0,
        'recall': (correct / total * 100) if total else 0,
    }


evaluation_summary = {
    'method1_round2_extraction': summarize_method(evaluation_rows, 'method1_predictions', 'method1_correct'),
    'traditional_method2_similarity_benchmark': summarize_method(evaluation_rows, 'traditional_method2_predictions', 'traditional_method2_correct'),
}
for model_name in bert_model_specs:
    evaluation_summary[f'{model_name}_method2'] = {
        'total': len(evaluation_rows),
        'predicted': sum(1 for row in evaluation_rows if row['bert_predictions'].get(model_name)),
        'coverage': (sum(1 for row in evaluation_rows if row['bert_predictions'].get(model_name)) / len(evaluation_rows) * 100) if evaluation_rows else 0,
        'correct': sum(1 for row in evaluation_rows if row['bert_correct'].get(model_name)),
        'accuracy': (
            sum(1 for row in evaluation_rows if row['bert_correct'].get(model_name)) /
            sum(1 for row in evaluation_rows if row['bert_predictions'].get(model_name)) * 100
        ) if sum(1 for row in evaluation_rows if row['bert_predictions'].get(model_name)) else 0,
        'recall': (sum(1 for row in evaluation_rows if row['bert_correct'].get(model_name)) / len(evaluation_rows) * 100) if evaluation_rows else 0,
    }

evaluation_table = pd.DataFrame([
    {
        'method': method_name,
        'total': summary['total'],
        'predicted': summary['predicted'],
        'correct': summary['correct'],
        'coverage': summary['coverage'],
        'accuracy': summary['accuracy'],
        'recall': summary['recall'],
    }
    for method_name, summary in evaluation_summary.items()
])

evaluation_table

evaluation_summary



{'method1_round2_extraction': {'total': 70,
  'predicted': 16,
  'coverage': 22.857142857142858,
  'correct': 9,
  'accuracy': 56.25,
  'recall': 12.857142857142856},
 'traditional_method2_similarity_benchmark': {'total': 70,
  'predicted': 64,
  'coverage': 91.42857142857143,
  'correct': 17,
  'accuracy': 26.5625,
  'recall': 24.285714285714285},
 'biobert_method2': {'total': 70,
  'predicted': 31,
  'coverage': 44.285714285714285,
  'correct': 1,
  'accuracy': 3.225806451612903,
  'recall': 1.4285714285714286},
 'pubmedbert_method2': {'total': 70,
  'predicted': 31,
  'coverage': 44.285714285714285,
  'correct': 1,
  'accuracy': 3.225806451612903,
  'recall': 1.4285714285714286},
 'scibert_method2': {'total': 70,
  'predicted': 31,
  'coverage': 44.285714285714285,
  'correct': 2,
  'accuracy': 6.451612903225806,
  'recall': 2.857142857142857}}

In [ ]:
entity_descriptions = {
    "sex": {
        "male": "A biological male individual or participant.",
        "female": "A biological female individual or participant."
    },
    "age": {
        "children": "Individuals in the pediatric or prepubescent age range.",
        "young adult": "Individuals in the post-pubescence phase, typically from late teens to early thirties.",
        "adult": "Physiologically mature individuals, typically in the middle-age range.",
        "elderly": "Older individuals, often associated with age-related physiological or musculoskeletal changes."
    },
    "level": {
        "athlete level": "Individuals with high physical performance, elite training status, or professional athletic background.",
        "non-athlete": "Members of the general population without specialized or high-intensity physical training."
    },
    "condition": {
        "patient": "Individuals exhibiting functional impairments, musculoskeletal discomfort, or biomechanical deviations such as flat feet or joint pain.",
        "healthy": "Asymptomatic individuals with optimal biomechanical function and no reported physical impairments."
    },
    "study_group": {
        "exposed group": "The cohort possessing the specific biomechanical characteristic or trait under observation.",
        "control group": "The baseline or reference cohort representing the standard physiological state for comparison."
    }
}

